# Stage 03: Supplementary real pairs from the Label Studio export  `[CPU]`
The clinician-corrected export is already a real `raw_asr -> gold_text` pair
(Whisper draft + human edit). Ingest it as supplementary real data (ViMedCSS stays
primary). Skipped automatically until the labeling workflow has produced rows.

In [ ]:
# --- CarePath stage bootstrap (short by design) ---
import importlib.util, os, subprocess, sys
from pathlib import Path

def _find(start):
    for d in [start, *start.parents]:
        if (d / 'pyproject.toml').exists() and (d / 'apps' / 'api' / 'carepath').exists():
            return d
    return None

REPO = _find(Path.cwd().resolve())
if REPO is None and importlib.util.find_spec('google.colab'):
    url = os.environ.get('CAREPATH_REPO_URL', 'https://github.com/truong-tt/carepath.git')
    tok = os.environ.get('CAREPATH_GITHUB_TOKEN') or os.environ.get('GITHUB_TOKEN')
    if tok and url.startswith('https://github.com/'):
        url = url.replace('https://', f'https://x-access-token:{tok}@')
    subprocess.run(['git', 'clone', url, '/content/carepath'], check=True)
    REPO = Path('/content/carepath')
assert REPO, 'Open this notebook from inside the CarePath repo.'
os.chdir(REPO); sys.path.insert(0, str(REPO / 'apps' / 'api'))

PROFILE = 'smoke'   # <<< set to 'full' for the real ViMedCSS run
from carepath.gec.notebook import init_stage
CTX = init_stage(PROFILE); P = CTX.paths; PROF = CTX.profile


In [ ]:
# Install the GEC training stack (idempotent; needed once per Colab runtime).
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[training]'])


In [ ]:
from pathlib import Path
export = 'data/labeling/training_transcripts.jsonl'
if Path(export).exists():
    CTX.run_step(['scripts/gec/make_labeled_pairs.py', '--input', export,
                  '--output', str(P.labeled_pairs), '--datastore', str(P.datastore),
                  '--retrieval-backend', PROF.retrieval_backend, '--resume'])
    CTX.save([str(P.labeled_pairs)])
else:
    print('No labeling export yet — see docs/vietnamese_labeling_guide.md. Skipping.')
